In [ ]:
!pip install transformers sentence-transformers faiss-cpu PyPDF2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 10.3 MB/s eta 0:00:00


In [ ]:
# ================================
# IMPORTS
# ================================
import PyPDF2
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline
from google.colab import files
import os


In [ ]:

# ================================
# LOAD MODELS
# ================================
print("Loading models...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)

print("Models loaded successfully!")

# ================================
# UPLOAD PDF
# ================================
print("Upload your PDF file:")
uploaded = files.upload()

# Get uploaded file name
file_name = list(uploaded.keys())[0]
print(f"Uploaded file: {file_name}")

# ================================
# LOAD PDF
# ================================
def load_pdf(file_path):
    text = ""
    with open(file_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            if page.extract_text():
                text += page.extract_text()
    return text

pdf_text = load_pdf(file_name)

if len(pdf_text.strip()) == 0:
    raise ValueError("PDF text extraction failed. Try another PDF.")

print("PDF loaded successfully!")

# ================================
# SPLIT TEXT INTO CHUNKS
# ================================
def split_text(text, chunk_size=300):
    words = text.split()
    chunks = [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

chunks = split_text(pdf_text)
print(f"Total chunks created: {len(chunks)}")

# ================================
# CREATE EMBEDDINGS + FAISS INDEX
# ================================
print("Creating embeddings...")
embeddings = embedder.encode(chunks)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

print("FAISS index created!")

# ================================
# QUESTION ANSWERING FUNCTION
# ================================
def ask_question(query):
    query_vec = embedder.encode([query])
    D, I = index.search(np.array(query_vec), k=5)

    context = " ".join([chunks[i] for i in I[0]])

    result = qa_pipeline(question=query, context=context)

    if len(result['answer']) < 5:
        return "Answer not found clearly in document."

    return result['answer']

# ================================
# INTERACTIVE LOOP
# ================================
print("\nAsk questions about your PDF (type 'exit' to stop):\n")

while True:
    query = input("Enter your question: ")

    if query.lower() == "exit":
        print("Exiting...")
        break

    answer = ask_question(query)
    print(f"Answer: {answer}\n")

Loading models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Models loaded successfully!
Upload your PDF file:


Saving Data Analytics and Data Science in Diagnosing and Managing Sleep Disorders Using Wearable Devices.pdf to Data Analytics and Data Science in Diagnosing and Managing Sleep Disorders Using Wearable Devices (2).pdf
Uploaded file: Data Analytics and Data Science in Diagnosing and Managing Sleep Disorders Using Wearable Devices (2).pdf
PDF loaded successfully!
Total chunks created: 14
Creating embeddings...
FAISS index created!

Ask questions about your PDF (type 'exit' to stop):

Enter your question: what is disorder they are talking about 
Answer: insomnia

Enter your question: explain the methodology 
Answer: More advanced techniques, including deep learning -based n eural networks

Enter your question: what is the Algorithm they used
Answer: convolutional neural networks

Enter your question: exit
Exiting...
